In [1]:
from src.ext.trapezoid_rnm import compute_trapz_rnm, OPT_CALL, OPT_PUT

In [3]:
from datetime import date

import numpy as np
import polars as pl
from src.risk_neutral_moments import DataSchema as DS

rng = np.random.default_rng(42)

SPOT       = 100.0
STOCKS     = [1, 2]
MONTHS     = [date(2023, m, 28) for m in range(1, 7)]
ATM_VOL    = 0.20          # base ATM vol
SKEW_SLOPE = 0.002         # per-unit-strike skew per month
TTM_DAYS   = 63            # ~1 quarter
RF         = 0.04

call_strikes = np.arange(105, 145, 5, dtype=float)   # 8 OTM calls
put_strikes  = np.arange(60,  100, 5, dtype=float)   # 8 OTM puts

rows = []
for stock_id in STOCKS:
    for month_idx, obs_date in enumerate(MONTHS):
        skew = SKEW_SLOPE * (month_idx + 1)

        for K in call_strikes:
            iv = ATM_VOL - skew * (K - SPOT)   # mild negative slope for calls
            iv = max(iv, 0.05)                  # floor at 5%
            rows.append({
                DS.stock_identifier: stock_id,
                DS.date: obs_date,
                DS.strike_price: K,
                DS.spot_price: SPOT,
                DS.implied_volatility: iv + rng.normal(0, 0.002),
                DS.option_type: DS.call_flag,
                DS.rf_rate: RF,
                DS.time_to_maturity: float(TTM_DAYS),
                # DS.moneyness: float(np.log(K / SPOT)),
            })

        for K in put_strikes:
            iv = ATM_VOL + skew * (SPOT - K)   # steeper smile for puts
            iv = max(iv, 0.05)
            rows.append({
                DS.stock_identifier: stock_id,
                DS.date: obs_date,
                DS.strike_price: K,
                DS.spot_price: SPOT,
                DS.implied_volatility: iv + rng.normal(0, 0.002),
                DS.option_type: DS.put_flag,
                DS.rf_rate: RF,
                DS.time_to_maturity: float(TTM_DAYS),
                # DS.moneyness:        float(np.log(K / SPOT)),
            })

options_data = pl.DataFrame(rows)
print(options_data.shape)
options_data.head(10)

(192, 8)


stock_id,date,strike_price,spot_price,implied_volatility,option_type,rf_rate,time_to_maturity
i64,date,f64,f64,f64,str,f64,f64
1,2023-01-28,105.0,100.0,0.190609,"""call""",0.04,63.0
1,2023-01-28,110.0,100.0,0.17792,"""call""",0.04,63.0
1,2023-01-28,115.0,100.0,0.171501,"""call""",0.04,63.0
1,2023-01-28,120.0,100.0,0.161881,"""call""",0.04,63.0
1,2023-01-28,125.0,100.0,0.146098,"""call""",0.04,63.0
1,2023-01-28,130.0,100.0,0.137396,"""call""",0.04,63.0
1,2023-01-28,135.0,100.0,0.130256,"""call""",0.04,63.0
1,2023-01-28,140.0,100.0,0.119368,"""call""",0.04,63.0
1,2023-01-28,60.0,100.0,0.279966,"""put""",0.04,63.0


In [3]:
from src.risk_neutral_moments import risk_neutral_moments

rnm = risk_neutral_moments(options_data, group_freq="1mo")
print(rnm)

shape: (12, 6)
┌──────────┬────────────┬──────────┬──────────┬──────────┬───────────┐
│ stock_id ┆ date       ┆ varQ     ┆ volQ     ┆ skewQ    ┆ kurtQ     │
│ ---      ┆ ---        ┆ ---      ┆ ---      ┆ ---      ┆ ---       │
│ i64      ┆ date       ┆ f64      ┆ f64      ┆ f64      ┆ f64       │
╞══════════╪════════════╪══════════╪══════════╪══════════╪═══════════╡
│ 1        ┆ 2023-01-01 ┆ 0.006525 ┆ 0.279816 ┆ 2.833574 ┆ 7.767629  │
│ 1        ┆ 2023-02-01 ┆ 0.007804 ┆ 0.306018 ┆ 3.588834 ┆ 15.822776 │
│ 1        ┆ 2023-03-01 ┆ 0.011067 ┆ 0.364415 ┆ 4.977932 ┆ 29.566126 │
│ 1        ┆ 2023-04-01 ┆ 0.017952 ┆ 0.464141 ┆ 5.554627 ┆ 31.27084  │
│ 1        ┆ 2023-05-01 ┆ 0.006525 ┆ 0.279816 ┆ 5.359712 ┆ 25.831815 │
│ …        ┆ …          ┆ …        ┆ …        ┆ …        ┆ …         │
│ 2        ┆ 2023-02-01 ┆ 0.007713 ┆ 0.304235 ┆ 3.622772 ┆ 16.057781 │
│ 2        ┆ 2023-03-01 ┆ 0.010989 ┆ 0.363138 ┆ 5.027237 ┆ 30.006172 │
│ 2        ┆ 2023-04-01 ┆ 0.02921  ┆ 0.592047 ┆ 5.403846 ┆ 26.

In [6]:
# Single group: 8 OTM calls + 8 OTM puts
spot       = 100.0
c_strikes  = np.arange(105, 145, 15, dtype=np.float64)
p_strikes  = np.arange(60,  100, 15, dtype=np.float64)
all_strikes= np.concatenate([c_strikes, p_strikes])

# Negatively-skewed smile: put vol > call vol
c_ivols    = np.linspace(0.18, 0.14, len(c_strikes))
p_ivols    = np.linspace(0.22, 0.32, len(p_strikes))
all_ivols  = np.concatenate([c_ivols, p_ivols])

all_flags  = np.array(
    [OPT_CALL] * len(c_strikes) + [OPT_PUT] * len(p_strikes),
    dtype=np.int32
)

indptr = np.array([0, len(all_strikes)], dtype=np.int64)

var, skew, kurt, rc = compute_trapz_rnm(
    strikes=all_strikes,
    ivols=all_ivols,
    flags=all_flags,
    spots=np.array([spot]),
    rf=np.array([0.04]),
    ttm=np.array([63.0 / 252.0]),
    indptr=indptr,
)

print(f"Skewness : {skew[0]:.4f}")
print(f"Variance : {var[0]:.6f}")
print(f"Kurtosis : {kurt[0]:.4f}")
print(f"Ann. Vol : {np.sqrt(var[0]) * np.sqrt(12):.4f}")
print(rc[0])



Skewness : 2.4759
Variance : 0.009138
Kurtosis : 5.6399
Ann. Vol : 0.3311
0


In [7]:
c_strikes

array([105., 120., 135.])

In [5]:
from src.risk_neutral_moments import DataSchema

ds = DataSchema()

def _minimal_options_df(n_stocks: int = 2, n_months: int = 3) -> pl.DataFrame:
    """
    Build a minimal synthetic Polars DataFrame that passes validation.

    Each (stock, month) has 8 OTM calls and 8 OTM puts.
    """
    rows = []
    spot = 100.0

    for s in range(n_stocks):
        for m in range(n_months):
            date = pl.date(2023, 1 + m, 28)
            for K in np.arange(105, 145, 5, dtype=float):
                rows.append({
                    ds.stock_identifier: s + 1,
                    ds.date: date,
                    ds.strike_price: K,
                    ds.spot_price: spot,
                    ds.implied_volatility: 0.20,
                    ds.option_type: "call",
                    ds.rf_rate: 0.02,
                    ds.time_to_maturity: 63.0,
                })
            for K in np.arange(60, 100, 5, dtype=float):
                rows.append({
                    ds.stock_identifier: s + 1,
                    ds.date: date,
                    ds.strike_price: K,
                    ds.spot_price: spot,
                    ds.implied_volatility: 0.20,
                    ds.option_type: "put",
                    ds.rf_rate: 0.02,
                    ds.time_to_maturity: 63.0,
                })

    return pl.DataFrame(rows)

In [6]:
df = _minimal_options_df()
out = risk_neutral_moments(df, group_freq="1mo")

InvalidOperationError: `round` operation not supported for dtype `object` (expected: date/datetime)

In [7]:
df



stock_id,date,strike_price,spot_price,implied_volatility,option_type,rf_rate,time_to_maturity
i64,object,f64,f64,f64,str,f64,f64
1,"2023-01-28 00:00:00.alias(""datetime"").strict_cast(Date).alias(""date"")",105.0,100.0,0.2,"""call""",0.02,63.0
1,"2023-01-28 00:00:00.alias(""datetime"").strict_cast(Date).alias(""date"")",110.0,100.0,0.2,"""call""",0.02,63.0
1,"2023-01-28 00:00:00.alias(""datetime"").strict_cast(Date).alias(""date"")",115.0,100.0,0.2,"""call""",0.02,63.0
1,"2023-01-28 00:00:00.alias(""datetime"").strict_cast(Date).alias(""date"")",120.0,100.0,0.2,"""call""",0.02,63.0
1,"2023-01-28 00:00:00.alias(""datetime"").strict_cast(Date).alias(""date"")",125.0,100.0,0.2,"""call""",0.02,63.0
…,…,…,…,…,…,…,…
2,"2023-03-28 00:00:00.alias(""datetime"").strict_cast(Date).alias(""date"")",75.0,100.0,0.2,"""put""",0.02,63.0
2,"2023-03-28 00:00:00.alias(""datetime"").strict_cast(Date).alias(""date"")",80.0,100.0,0.2,"""put""",0.02,63.0
2,"2023-03-28 00:00:00.alias(""datetime"").strict_cast(Date).alias(""date"")",85.0,100.0,0.2,"""put""",0.02,63.0
